# PSE-823 Advanced Process Dynamics and Control
## Assignment 1: Open-loop dynamics of a first-order and a second-order system

**Due:** 30 October 2026. **Reference:** Coughanowr and LeBlanc, Chapters 5 and 7. **Tools:** `numpy`, `scipy`, `matplotlib`, `python-control`.

Task 1 of each Part (the derivations) must be done **by hand**; submit clear photos or scans of your handwritten notes (one PDF; typed derivations are not accepted) together with this notebook, which you may submit as the `.ipynb` **or** as a PDF export. This notebook holds Tasks 2 and 3 of each Part. In the first code cell of each Part, **enter the numbers from your hand derivation**; the code then checks them against the numerical results. Every figure needs labelled axes with units. Every written answer must quote numbers your own code produced.

Before submitting: *Runtime > Restart and run all*, then check that nothing errors and all outputs are visible. Submit both files on the course **Microsoft Teams** class (Assignments tab) by **30 October 2026**.

In [ ]:
# On Colab, uncomment the next line once:
# !pip -q install control

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import find_peaks
import control as ct

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "axes.axisbelow": True})

REG_LAST_DIGIT = 0     # <-- replace with the last digit of YOUR Reg #
d = REG_LAST_DIGIT
assert 0 <= d <= 9

---
# Part A: First-order system, liquid-level tank with a square-root valve (22 marks)

**Source:** Coughanowr, Sec. 5.2 (Eqs. 5.30-5.38) and Prob. 5.2.

A tank of cross-section $A = 1.0 + 0.2\,d$ m$^2$ and height 6.0 m drains through $q_o = C\sqrt{h}$ (Eq. 5.30). At the operating point $q_s = 0.020$ m$^3$/s and $h_s = 2.0$ m (water, constant density).

### A1. By hand (7 marks) - on paper, not in this notebook
(a) From the mass balance (Eq. 5.31) find $C$, linearize about $h_s$ and obtain $H(s)/Q(s) = R_1/(\tau s + 1)$ with numerical $R_1$, $\tau$ and units. (b) Write the linear step response for $\Delta q = f\,q_s$. (c) For steps of $+10$ %, $+60$ %, $-60$ % in $q$, compute the exact final level $(q/C)^2$ and the linear-model final level. (d) Compute the true time constant $2A\sqrt{h}/C$ at each new final level.

In [ ]:
# ---- Part A setup: enter YOUR hand results from A1 ----
A      = 1.0 + 0.2*d          # m^2
hs, qs = 2.0, 0.020           # m, m^3/s
h_tank = 6.0                  # m
steps  = {"+10%": 0.10, "+60%": 0.60, "-60%": -0.60}

C   = ...                     # m^2.5/s   from A1(a)
R1  = ...                     # s/m^2     from A1(a)
tau = ...                     # s         from A1(a)
h_inf_hand   = {"+10%": ..., "+60%": ..., "-60%": ...}   # m, exact final levels, A1(c)
tau_new_hand = {"+10%": ..., "+60%": ..., "-60%": ...}   # s, true tau at the new levels, A1(d)

### A2. Code: linear versus nonlinear (9 marks)
1. Write `h_linear(t, frac)` (your A1(b)) and `h_nonlinear(t, frac)` (integrate Eq. 5.32 with `solve_ivp`; guard the square root against negative $h$).
2. Plot both for the three steps, three panels side by side, over $0 \le t \le 6\tau$. Mark $h = 0$ and the tank height.
3. Check your linear function against `ct.tf` and `ct.step_response` (print the largest difference).
4. Print one table row per step: your hand final level (exact and linear), the maximum $|h_\text{nl} - h_\text{lin}|$, that error as % of the change, the *apparent time constant* (time for the nonlinear response to cover 63.2 % of its own change) and your hand $\tau$ from A1(d). Also print the difference between the ODE value at $12\tau$ and your hand exact final level.

In [ ]:
def h_linear(t, frac):
    """Linearized model, step of size frac*qs in inlet flow."""
    return ...   # TODO: your A1(b)

def h_nonlinear(t, frac):
    """Solution of A dh/dt = q - C sqrt(h)."""
    ...          # TODO: solve_ivp, return the array h(t)

t = np.linspace(0, 6*tau, 1201)

# TODO: three-panel figure (nonlinear vs linearized), mark h = 0 and h = h_tank

# TODO: python-control cross-check of the +10 % linear case

In [ ]:
def t_apparent(t, h, h_new):
    ...   # TODO: time at which h has covered 63.2 % of its change (np.interp)

# TODO: loop over `steps` and print the table (hand final levels, max error, error %, tau_app, hand tau)
# TODO: compare the ODE value at 12 tau with your hand exact final level

### A3. Code and interpret (6 marks)
1. **Code (3).** Plot $\tau(h_s) = 2A\sqrt{h_s}/C$ for $0.2 \le h_s \le 6$ m and mark the original level and the three new final levels. By repeated simulation find the largest step *up* and *down* (fraction of $q_s$, 1 % resolution) for which the linearization error stays within 5 % of the total change.
2. **Interpret (3).** In at most six sentences, quoting your numbers: for which steps is the linear model acceptable and why does its error grow with step size; what does it predict for the $-60$ % step and why is that impossible, and what does the nonlinear model do; does the $+60$ % step overflow the tank, and is the linear prediction on the safe or unsafe side for a high-level alarm?

*Write your answer in the cell below.*

In [ ]:
hh = np.linspace(0.2, 6, 300)
# TODO: plot tau(h_s) and mark the original and the three new final levels

def pct_err(f, n=601):
    ...   # TODO: max |h_nl - h_lin| as % of the exact change, for a step f

# TODO: scan f and report the largest +/- steps with error <= 5 %

**A3 interpretation (answer here, at most six sentences):**

*(double-click to edit)*

---
# Part B: Second-order system, the U-tube manometer (22 marks)

**Source:** Coughanowr, Sec. 7.1 (Eqs. 7.1-7.12, 7.18-7.28) and Prob. 7.5.

A U-tube manometer with fluid-column length $L = 1.0 + 0.1\,d$ m, water at 20 $^\circ$C ($\rho = 998$ kg/m$^3$, $\mu = 1.00$ mPa$\cdot$s, $g = 9.81$ m/s$^2$), laminar flow, $\beta = 4/3$, is suddenly subjected to $\Delta P/(\rho g) = 0.20$ m of water. Tube diameters: $D = 1.0,\ 1.5,\ 2.5$ mm.

### B1. By hand (7 marks) - on paper, not in this notebook
(a) From Eq. 7.3 obtain Eq. 7.4 and the standard form $\tau^2 Y'' + 2\zeta\tau Y' + Y = X(t)$; derive $\tau$ and $\zeta$ in terms of $L, \mu, \rho, D, g$ (check the square roots yourself). (b) Compute $\tau$ and $\zeta$ for the three tubes and classify each. (c) For the underdamped tube compute the overshoot, decay ratio and period (Eqs. 7.25-7.28). (d) Find the diameter that gives $\zeta = 0.7$.

In [ ]:
# ---- Part B setup: enter YOUR hand results from B1 ----
g, rho, mu = 9.81, 998.0, 1.0e-3
L  = 1.0 + 0.1*d                 # m
M  = 0.20                        # m, size of the step in dP/(rho g)
Ds = np.array([1.0, 1.5, 2.5])*1e-3   # m

tau2 = ...                       # s,  B1(b)
zeta = np.array([..., ..., ...]) # -,  B1(b), one per diameter
iu, io = int(np.argmin(zeta)), int(np.argmax(zeta))   # underdamped / most overdamped tube
os_hand, decay_hand, T_hand = ..., ..., ...           # B1(c)  (T in s)
D_design = ...                   # m,  B1(d)

### B2. Code: three solutions and measurements (9 marks)
1. For each tube compute the step response $h(t)$ (step size $M$) three ways: (i) **analytic**, one function `y_step(t, zeta, tau)` returning the unit step response for all three damping cases (Eqs. 7.18, 7.20, 7.21); (ii) **ODE**, `solve_ivp` on the two-state form (keep the velocity); (iii) **python-control**, `ct.tf` and `ct.step_response`.
2. Plot the three tubes on one figure for $0 \le t \le 8$ s (analytic curves, the other two methods as markers) and print the largest difference between the methods for each tube.
3. For the underdamped tube **measure from the simulated curve** the overshoot, decay ratio and period, and print them next to your hand values from B1(c). Report the 5 % response time of all three tubes.

In [ ]:
def y_step(t, z, tau):
    """Unit step response of 1/(tau^2 s^2 + 2 z tau s + 1), all three damping cases."""
    ...   # TODO (hint: np.arctan2(w, z) avoids a quadrant error in Eq. 7.18)

t2 = np.linspace(0, 8, 8001)
res = {}
# TODO: for each D compute (i) analytic, (ii) solve_ivp (keep the velocity state), (iii) ct.step_response
#       store y and dy/dt in `res[D]`, plot, and print the largest differences

In [ ]:
def t5(y, ss):
    """5 % response time: last time the curve is outside +/-5 % of its final value."""
    ...   # TODO

y = res[Ds[iu]]["y"]
# TODO: measure overshoot, decay ratio and period of the underdamped curve (hint: scipy.signal.find_peaks)
# TODO: print a measured-vs-hand table; print the 5 % response time of all three tubes

### B3. Code and interpret (6 marks)
1. **Code (3).** (a) For $0.05 \le \zeta \le 3$ compute the overshoot (%) and the 5 % response time divided by $\tau$ (use `y_step` on a fine normalized time grid); plot both against $\zeta$ and report the $\zeta$ that minimizes the response time. Simulate your B1(d) design diameter and report its overshoot and 5 % response time. (b) Compute the peak Reynolds number $\rho\bar V_\text{max}D/\mu$ for each tube from your simulated velocity ($\bar V = \tfrac12|dh/dt|$). (c) For the most overdamped tube use Eqs. 7.23-7.24 to split the system into two first-order lags, verify with `ct.series` that they reproduce the step response, and compare with a single lag $\tau_1$.
2. **Interpret (3).** In at most six sentences, quoting your numbers: which tube would you fit if the reading must be within 5 % of the true value in the shortest time, and why (relate to the sweep); which of the two lags makes the overdamped tube slow and why is the other almost invisible; does the laminar assumption hold for all three tubes?

*Write your answer in the cell below.*

In [ ]:
zs = np.linspace(0.05, 3, 300)
tn = np.linspace(0, 30, 30001)               # time in units of tau
# TODO (a): overshoot (%) and 5 % response time / tau for every zeta; plot both; report the best zeta
# TODO (a): simulate the design diameter D_design; report overshoot and 5 % response time
# TODO (b): peak Reynolds number for each tube from res[D]["yprime"]
# TODO (c): tau1, tau2 (Eqs. 7.23-7.24) for the most overdamped tube; ct.series check; plot against one lag tau1

**B3 interpretation (answer here, at most six sentences):**

*(double-click to edit)*

---
# Part C: Presentation and reproducibility (6 marks)

The notebook runs top to bottom without errors after *Restart and run all* (3); every figure has labelled axes, units and a legend where needed (2); derivations are legible and code is commented with the symbols used in the text (1).